# Phase 1: TF-IDF N-grams & Traditional ML Baseline Models
### Classification of Human-Written vs. AI-Generated Text
**National Institute of Technology Sikkim** — Dept. of Computer Science & Engineering  
*Authors:* Pravesh Gedon Sharma (B240019) & Ankit Rawat (B320032)

---
### 🎯 Objectives of this Notebook:
1. **Implement Dual TF-IDF Representation** (Section 4.3):
   - **Word n-grams** (1 to 2): Captures vocabulary choices, collocations, and phrase usage.
   - **Character n-grams** (3 to 5 with  analyzer): Captures sub-word morphology, character repetition, punctuation habits, and formatting quirks across word boundaries.
2. **Train Traditional ML Classifiers** (Section 4.5):
   - **Logistic Regression** (, )
   - **Linear Support Vector Machine (Linear SVM / LinearSVC)**
   - **Multinomial Naive Bayes (Fast Lexical Baseline)**
3. **Feature Leakage Control**: Fit vectorizers and scalers exclusively on training splits, then transform validation/test splits.
4. **Model Evaluation & Diagnostic Benchmarking** (Section 6.3):
   - Calculate Accuracy, Precision, Recall, F1-Score, and ROC-AUC.
   - Plot Confusion Matrices and ROC Curves.
   - Extract Top Informative N-grams (most indicative human vs. AI tokens).

## 1. Setup & Imports

In [ ]:
import sys
import os
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import hstack
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report

from src.preprocessing import clean_text_for_tfidf
from src.evaluate import compute_classification_metrics, plot_confusion_matrix, plot_roc_curves

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
os.makedirs("../models", exist_ok=True)
os.makedirs("../results", exist_ok=True)
print("Environment initialized.")

## 2. Load and Prepare Training & Validation Data
To balance rapid iterative training with statistical reliability, we sample a stratified subset of 20,000 documents (10,000 Human and 10,000 AI) for training, and evaluate on both an in-distribution validation split and test samples.

In [ ]:
TRAIN_PATH = "../train.csv" if os.path.exists("../train.csv") else "train.csv"
TEST_PATH = "../test.csv" if os.path.exists("../test.csv") else "test.csv"

df_raw = pd.read_csv(TRAIN_PATH)
print(f"Raw dataset shape: {df_raw.shape}")

# Balanced sample for baseline model training
SAMPLE_SIZE = 10000  # 10k human + 10k AI
df_train_sub = df_raw.groupby("generated", group_keys=False).apply(
    lambda x: x.sample(SAMPLE_SIZE, random_state=42)
).reset_index(drop=True)

print("Class distribution in experiment subset:")
print(df_train_sub["generated"].value_counts())

# Stratified Train/Val split (80/20)
train_texts, val_texts, y_train, y_val = train_test_split(
    df_train_sub["text"].values,
    df_train_sub["generated"].values,
    test_size=0.20,
    random_state=42,
    stratify=df_train_sub["generated"].values
)

print(f"Train samples: {len(train_texts):,}")
print(f"Validation samples: {len(val_texts):,}")

## 3. Dual TF-IDF Feature Extraction (Section 4.3)
According to Section 4.3 of our report:
- **Word n-grams (1, 2)**: , , , 
- **Char n-grams (3, 5)**: , , , , 
- Stored as SciPy CSR arrays and concatenated horizontally using .

In [ ]:
print("Vectorizing Word N-grams (1, 2)...")
word_vec = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=25000,
    min_df=2,
    sublinear_tf=True,
    preprocessor=clean_text_for_tfidf
)
X_train_word = word_vec.fit_transform(train_texts)
X_val_word = word_vec.transform(val_texts)
print(f"Word TF-IDF shape: {X_train_word.shape}")

print("
Vectorizing Character N-grams (3, 5) with char_wb...")
char_vec = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    max_features=50000,
    min_df=3,
    sublinear_tf=True
)
X_train_char = char_vec.fit_transform(train_texts)
X_val_char = char_vec.transform(val_texts)
print(f"Char TF-IDF shape: {X_train_char.shape}")

# Horizontally stack word and character sparse feature spaces
X_train_tfidf = hstack([X_train_word, X_train_char]).tocsr()
X_val_tfidf = hstack([X_val_word, X_val_char]).tocsr()
print(f"Combined Sparse Matrix Shape: {X_train_tfidf.shape}")

## 4. Model 1: Multinomial Naive Bayes (Fast Probabilistic Baseline)

In [ ]:
nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train_tfidf, y_train)

y_pred_nb = nb_model.predict(X_val_tfidf)
y_prob_nb = nb_model.predict_proba(X_val_tfidf)[:, 1]

metrics_nb = compute_classification_metrics(y_val, y_pred_nb, y_prob_nb)
print("=== Multinomial Naive Bayes Metrics ===")
for k, v in metrics_nb.items():
    print(f"{k.capitalize()}: {v:.4f}")

## 5. Model 2: Logistic Regression (L2 Regularized Baseline)
Sparse TF-IDF features typically show strong linear separability.

In [ ]:
lr_model = LogisticRegression(C=1.5, max_iter=1000, solver="lbfgs", random_state=42)
lr_model.fit(X_train_tfidf, y_train)

y_pred_lr = lr_model.predict(X_val_tfidf)
y_prob_lr = lr_model.predict_proba(X_val_tfidf)[:, 1]

metrics_lr = compute_classification_metrics(y_val, y_pred_lr, y_prob_lr)
print("=== Logistic Regression Metrics ===")
for k, v in metrics_lr.items():
    print(f"{k.capitalize()}: {v:.4f}")

## 6. Model 3: Linear Support Vector Machine (Linear SVM)
Using  wrapped around  so we obtain calibrated posterior probabilities for ROC-AUC.

In [ ]:
base_svm = LinearSVC(C=1.0, max_iter=2000, random_state=42)
svm_model = CalibratedClassifierCV(base_svm, cv=3)
svm_model.fit(X_train_tfidf, y_train)

y_pred_svm = svm_model.predict(X_val_tfidf)
y_prob_svm = svm_model.predict_proba(X_val_tfidf)[:, 1]

metrics_svm = compute_classification_metrics(y_val, y_pred_svm, y_prob_svm)
print("=== Linear SVM (Calibrated) Metrics ===")
for k, v in metrics_svm.items():
    print(f"{k.capitalize()}: {v:.4f}")

## 7. Comparative Performance Table & Visualizations

In [ ]:
benchmark_df = pd.DataFrame([
    {"Model": "Multinomial Naive Bayes", **metrics_nb},
    {"Model": "Logistic Regression", **metrics_lr},
    {"Model": "Linear SVM", **metrics_svm}
])

display(benchmark_df.round(4))
benchmark_df.to_csv("../results/baseline_ml_metrics.csv", index=False)

# Plot ROC curves side-by-side with confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_confusion_matrix(y_val, y_pred_nb, title="NB Confusion Matrix", ax=axes[0])
plot_confusion_matrix(y_val, y_pred_lr, title="Logistic Regression Confusion Matrix", ax=axes[1])
plot_confusion_matrix(y_val, y_pred_svm, title="Linear SVM Confusion Matrix", ax=axes[2])
plt.tight_layout()
plt.show()

# Plot ROC comparison
plt.figure(figsize=(8, 6))
probs_dict = {
    "Naive Bayes": y_prob_nb,
    "Logistic Regression": y_prob_lr,
    "Linear SVM": y_prob_svm
}
plot_roc_curves(probs_dict, y_val)
plt.show()

## 8. Interpretability: Most Informative N-grams
Examining coefficients from the Logistic Regression model allows us to observe which tokens and character n-grams are strongly associated with human writing vs. machine generation.

In [ ]:
feature_names = np.array(word_vec.get_feature_names_out().tolist() + char_vec.get_feature_names_out().tolist())
coefs = lr_model.coef_[0]

top_human_idx = np.argsort(coefs)[:15]
top_ai_idx = np.argsort(coefs)[-15:][::-1]

tokens_df = pd.DataFrame({
    "Human Predictive Feature": feature_names[top_human_idx],
    "Human Coef": coefs[top_human_idx],
    "AI Predictive Feature": feature_names[top_ai_idx],
    "AI Coef": coefs[top_ai_idx]
})
print("Top 15 Most Discriminative Features:")
display(tokens_df)

## 9. Save Best Vectorizers and Models for the Streamlit App

In [ ]:
joblib.dump(word_vec, "../models/word_tfidf_vectorizer.pkl")
joblib.dump(char_vec, "../models/char_tfidf_vectorizer.pkl")
joblib.dump(lr_model, "../models/logistic_regression_baseline.pkl")
joblib.dump(svm_model, "../models/linear_svm_baseline.pkl")
print("Model artifacts and vectorizers saved successfully into .")

## 10. Key Takeaways & Transition to Notebook 04
1. **High Discrimination via Combined N-grams**: Character n-grams () paired with word TF-IDF provide strong baseline accuracy and high ROC-AUC.
2. **Interpretability**: Distinct lexical signatures (e.g. overused introductory transitions vs. organic informal/human conversational markers) are confirmed by feature coefficients.
3. **Next Step**: In , we incorporate **Sentence-BERT semantic representations** and concatenate them with stylometric features for the Hybrid Feature-Fusion model.